# Kitchen-Level P&L Dashboard - Interactive Notebook

This notebook provides an interactive version of the Kitchen-Level Profit & Loss Dashboard with comprehensive financial analysis, filtering capabilities, and visualizations.

## Features:
- Interactive data filtering by store, month, zone, and revenue cohorts
- Financial performance metrics and KPIs
- P&L breakdown with revenue, costs, and profitability analysis
- Interactive charts and visualizations
- Trend analysis and comparative insights

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

In [ ]:
# Load and Preprocess Kitchen Data
def load_kitchen_data():
    """Load and preprocess the kitchen dashboard data."""
    # Load data with first row as headers
    df = pd.read_excel('dummy_data.xlsx', header=0)
    
    # Check if the actual headers are in the first row of data
    if df.iloc[0, 0] == 'MONTH':
        # Use the first row as column names and drop it
        df.columns = df.iloc[0]
        df = df.drop(df.index[0]).reset_index(drop=True)
        
        # Reset column names to remove any index references
        df.columns.name = None
    
    # Ensure numeric columns are properly typed
    numeric_columns = ['ORDER COUNT', 'CART SALES', 'DISCOUNT', 'NET REVENUE', 
                      'IDEAL FOOD COST', 'GROSS MARGIN', 'KITCHEN EBITDA', 'VARIANCE']
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Calculate additional metrics
    df['GROSS_MARGIN_PCT'] = (df['GROSS MARGIN'] / df['NET REVENUE'] * 100).round(2)
    df['EBITDA_PCT'] = (df['KITCHEN EBITDA'] / df['NET REVENUE'] * 100).round(2)
    df['FOOD_COST_PCT'] = (df['IDEAL FOOD COST'] / df['NET REVENUE'] * 100).round(2)
    df['DISCOUNT_PCT'] = (df['DISCOUNT'] / df['CART SALES'] * 100).round(2)
    df['AVG_ORDER_VALUE'] = (df['NET REVENUE'] / df['ORDER COUNT']).round(2)
    
    # Convert MONTH to datetime for better sorting
    df['MONTH_DATE'] = pd.to_datetime(df['MONTH'], format='%b-%Y')
    df = df.sort_values('MONTH_DATE')
    
    return df

# Load the data
df_kitchen = load_kitchen_data()
print(f"✅ Kitchen data loaded successfully!")
print(f"📊 Dataset shape: {df_kitchen.shape}")
print(f"📅 Date range: {df_kitchen['MONTH'].min()} to {df_kitchen['MONTH'].max()}")
print(f"🏪 Total stores: {df_kitchen['STORE'].nunique()}")
print(f"🏙️ Total cities: {df_kitchen['CITY'].nunique()}")
print(f"💰 Total revenue: ₹{df_kitchen['NET REVENUE'].sum():,.0f}")

# Display first few rows
display(df_kitchen.head())

In [ ]:
# Create Interactive Controls for Kitchen Dashboard
def get_kitchen_filter_options(df):
    """Get available options for kitchen dashboard filters."""
    stores = ['All'] + sorted(df['STORE'].unique().tolist())
    months = ['All'] + sorted(df['MONTH'].unique().tolist(), 
                             key=lambda x: pd.to_datetime(x, format='%b-%Y'))
    zones = ['All'] + sorted(df['ZONE MAPPING'].unique().tolist())
    cohorts = ['All'] + sorted(df['REVENUE COHORT'].unique().tolist())
    
    return stores, months, zones, cohorts

# Get filter options
stores, months, zones, cohorts = get_kitchen_filter_options(df_kitchen)

# Create interactive widgets
store_widget = widgets.Dropdown(
    options=stores,
    value="All",
    description='Store:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

month_widget = widgets.Dropdown(
    options=months,
    value="All",
    description='Month:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

zone_widget = widgets.Dropdown(
    options=zones,
    value="All",
    description='Zone:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

cohort_widget = widgets.Dropdown(
    options=cohorts,
    value="All",
    description='Revenue Cohort:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# Group widgets
kitchen_controls = widgets.VBox([
    widgets.HTML("<h3>🎛️ Kitchen Dashboard Controls</h3>"),
    widgets.HBox([store_widget, month_widget]),
    widgets.HBox([zone_widget, cohort_widget])
])

display(kitchen_controls)
print("✅ Kitchen dashboard controls created!")

In [ ]:
# Kitchen Dashboard Functions and Interactive Display
def filter_kitchen_data(df, store="All", month="All", zone="All", cohort="All"):
    """Filter kitchen data based on selections."""
    filtered_df = df.copy()
    
    if store != 'All':
        filtered_df = filtered_df[filtered_df['STORE'] == store]
    if month != 'All':
        filtered_df = filtered_df[filtered_df['MONTH'] == month]
    if zone != 'All':
        filtered_df = filtered_df[filtered_df['ZONE MAPPING'] == zone]
    if cohort != 'All':
        filtered_df = filtered_df[filtered_df['REVENUE COHORT'] == cohort]
    
    return filtered_df

def create_kitchen_kpis(df):
    """Create KPI metrics for kitchen dashboard."""
    total_revenue = df['NET REVENUE'].sum()
    total_orders = df['ORDER COUNT'].sum()
    avg_order_value = total_revenue / total_orders if total_orders > 0 else 0
    total_ebitda = df['KITCHEN EBITDA'].sum()
    ebitda_margin = (total_ebitda / total_revenue * 100) if total_revenue > 0 else 0
    gross_margin = df['GROSS MARGIN'].sum()
    gross_margin_pct = (gross_margin / total_revenue * 100) if total_revenue > 0 else 0
    
    kpis_html = f"""
    <div style="display: flex; gap: 15px; margin: 20px 0; flex-wrap: wrap;">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 15px; border-radius: 10px; text-align: center; flex: 1; min-width: 200px;">
            <h3 style="margin: 0; font-size: 1.8em;">₹{total_revenue:,.0f}</h3>
            <p style="margin: 5px 0 0 0;">Total Revenue</p>
        </div>
        <div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); 
                    color: white; padding: 15px; border-radius: 10px; text-align: center; flex: 1; min-width: 200px;">
            <h3 style="margin: 0; font-size: 1.8em;">{total_orders:,}</h3>
            <p style="margin: 5px 0 0 0;">Total Orders</p>
        </div>
        <div style="background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%); 
                    color: white; padding: 15px; border-radius: 10px; text-align: center; flex: 1; min-width: 200px;">
            <h3 style="margin: 0; font-size: 1.8em;">₹{avg_order_value:.0f}</h3>
            <p style="margin: 5px 0 0 0;">Avg Order Value</p>
        </div>
        <div style="background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%); 
                    color: white; padding: 15px; border-radius: 10px; text-align: center; flex: 1; min-width: 200px;">
            <h3 style="margin: 0; font-size: 1.8em;">{ebitda_margin:.1f}%</h3>
            <p style="margin: 5px 0 0 0;">EBITDA Margin</p>
        </div>
        <div style="background: linear-gradient(135deg, #fa709a 0%, #fee140 100%); 
                    color: white; padding: 15px; border-radius: 10px; text-align: center; flex: 1; min-width: 200px;">
            <h3 style="margin: 0; font-size: 1.8em;">{gross_margin_pct:.1f}%</h3>
            <p style="margin: 5px 0 0 0;">Gross Margin %</p>
        </div>
    </div>
    """
    return kpis_html

def create_revenue_trend(df):
    """Create revenue trend chart."""
    monthly_data = df.groupby('MONTH').agg({
        'NET REVENUE': 'sum',
        'ORDER COUNT': 'sum',
        'KITCHEN EBITDA': 'sum'
    }).reset_index()
    
    # Sort by month chronologically
    monthly_data['MONTH_DATE'] = pd.to_datetime(monthly_data['MONTH'], format='%b-%Y')
    monthly_data = monthly_data.sort_values('MONTH_DATE')
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Revenue Trend', 'Order Count Trend', 'EBITDA Trend', 'Revenue Breakdown'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"type": "pie"}]]
    )
    
    # Revenue trend
    fig.add_trace(
        go.Scatter(x=monthly_data['MONTH'], y=monthly_data['NET REVENUE'],
                  mode='lines+markers', name='Revenue', line=dict(color='#1f77b4')),
        row=1, col=1
    )
    
    # Order trend
    fig.add_trace(
        go.Scatter(x=monthly_data['MONTH'], y=monthly_data['ORDER COUNT'],
                  mode='lines+markers', name='Orders', line=dict(color='#ff7f0e')),
        row=1, col=2
    )
    
    # EBITDA trend
    fig.add_trace(
        go.Scatter(x=monthly_data['MONTH'], y=monthly_data['KITCHEN EBITDA'],
                  mode='lines+markers', name='EBITDA', line=dict(color='#2ca02c')),
        row=2, col=1
    )
    
    # Revenue breakdown pie chart
    revenue_breakdown = df.groupby('REVENUE COHORT')['NET REVENUE'].sum()
    fig.add_trace(
        go.Pie(labels=revenue_breakdown.index, values=revenue_breakdown.values, name="Revenue Breakdown"),
        row=2, col=2
    )
    
    fig.update_layout(height=600, title_text="Kitchen Performance Overview", title_x=0.5)
    return fig

# Interactive Kitchen Dashboard
kitchen_output = widgets.Output()

def update_kitchen_dashboard(*args):
    """Update kitchen dashboard based on widget values."""
    with kitchen_output:
        clear_output(wait=True)
        
        # Get current filter values
        store_val = store_widget.value
        month_val = month_widget.value
        zone_val = zone_widget.value
        cohort_val = cohort_widget.value
        
        # Apply filters
        filtered_df = filter_kitchen_data(df_kitchen, store_val, month_val, zone_val, cohort_val)
        
        if len(filtered_df) == 0:
            display(HTML("<h3 style='color: red;'>⚠️ No data matches the selected filters.</h3>"))
            return
        
        # Display KPIs
        kpis_html = create_kitchen_kpis(filtered_df)
        display(HTML(f"<h2>📊 Key Performance Indicators</h2>{kpis_html}"))
        
        # Display revenue trends
        fig_trends = create_revenue_trend(filtered_df)
        display(HTML("<h2>📈 Performance Trends</h2>"))
        fig_trends.show()
        
        # Display detailed data table
        display(HTML("<h3>📋 Detailed Data</h3>"))
        summary_cols = ['MONTH', 'STORE', 'CITY', 'NET REVENUE', 'ORDER COUNT', 
                       'KITCHEN EBITDA', 'GROSS MARGIN', 'AVG_ORDER_VALUE', 'EBITDA_PCT']
        display(filtered_df[summary_cols].style.format({
            'NET REVENUE': '₹{:,.0f}',
            'KITCHEN EBITDA': '₹{:,.0f}',
            'GROSS MARGIN': '₹{:,.0f}',
            'AVG_ORDER_VALUE': '₹{:.0f}',
            'EBITDA_PCT': '{:.1f}%'
        }))

# Attach observers
store_widget.observe(update_kitchen_dashboard, names='value')
month_widget.observe(update_kitchen_dashboard, names='value')
zone_widget.observe(update_kitchen_dashboard, names='value')
cohort_widget.observe(update_kitchen_dashboard, names='value')

# Initial update
update_kitchen_dashboard()

# Display output
display(kitchen_output)

print("✅ Interactive Kitchen Dashboard created! Use the controls above to filter and explore the data.")